# L5b Lab: Maximum-Flow Sensitivity and Bottlenecks

Suppose three workers are available to complete four tasks. Each worker can initially accept one assignment, and each task needs one worker. If one worker can accept a second assignment, can we complete more work? What happens if that worker becomes unavailable instead?

The [L5a lecture](../L5a/CHEME-5800-L5a-Lecture-MaximumFlowProblems-Fall-2026.ipynb) and [worked example](../L5a/CHEME-5800-L5a-WorkedExample-MaximumFlow-Fall-2026.ipynb) represented assignments as flow through a network. One unit of flow from the source to the sink corresponds to one completed assignment. Here, we study __capacity sensitivity__: how changing edge capacities changes the maximum number of assignments the network can complete.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * **Establish a maximum-flow baseline:** Compute the maximum number of assignments for the supplied network and check the returned flow against capacity, conservation, and source–sink balance constraints.
> * **Evaluate capacity changes:** Predict and compute the effects of increasing a worker's assignment capacity and making that worker unavailable, comparing each scenario with the same baseline.
> * **Explain network bottlenecks:** Use feasible assignment routes and source–sink cuts to explain how the worker and task capacities limit completed work.

In this lab, we use the supplied code to establish the baseline, increase the capacity available to the worker at node `3`, and then examine that worker's outage in a separate copy of the baseline network. Before each intervention, we will predict the outcome; afterward, we will check the computed flow and explain what limits the number of assignments.

Let's get started!


___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course library, the lab's graph-building and flow-validation helpers, and the packages used here.

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course library documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/) for the functions and types used here.

The network is supplied in the local [Workers-Tasks-Bipartite.edgelist file](data/Workers-Tasks-Bipartite.edgelist). Each record lists an edge's source node, target node, cost, lower capacity, and upper capacity. We use the capacity bounds to limit the flow; the cost field is not part of this maximum-flow objective.


___

## Task 1: Build and validate the baseline network

In this task, we build the worker–task network and establish a validated maximum-flow baseline for comparing the two interventions. The [edge-list file](data/Workers-Tasks-Bipartite.edgelist) uses the following node identifiers:

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr><th style="text-align: left;">Nodes</th><th style="text-align: left;">Role in the assignment model</th></tr>
</thead>
<tbody>
<tr><td style="text-align: left;">1</td><td style="text-align: left;">The source supplies flow to allocate work.</td></tr>
<tr><td style="text-align: left;">2–4</td><td style="text-align: left;">Three workers receive assignments from the source.</td></tr>
<tr><td style="text-align: left;">5–8</td><td style="text-align: left;">Four tasks receive flow from the workers assigned to them.</td></tr>
<tr><td style="text-align: left;">9–12</td><td style="text-align: left;">Task-completion nodes pass completed work toward the sink. Nodes 9, 10, 11, and 12 correspond to tasks 5, 6, 7, and 8.</td></tr>
<tr><td style="text-align: left;">13</td><td style="text-align: left;">The sink collects the total completed work.</td></tr>
</tbody>
</table>

Every worker is connected to every task in this dataset. All edges have a lower flow bound of zero and an upper capacity of one assignment. The source-to-worker edges limit each worker to one assignment, while the task-to-completion edges limit the total flow through each task to one unit.

For example, one unit of flow along the route `1 → 2 → 5 → 9 → 13` represents worker `2` completing task `5`. Node `9` passes that same unit toward the sink; it does not represent another assignment. Conservation requires the flow entering each intermediate node to equal the flow leaving it.

We will use these capacities and node roles to build the graph and check its baseline maximum flow.


### Build the graph and compute the flow

[The `build_sensitivity_graph(...)` function](docs/build_sensitivity_graph.md) reads the edge list and returns the graph stored in `baseline_graph`. We specify node `1` as the source and node `13` as the sink. The graph's `capacity` dictionary stores each directed edge `(u, v)` with its `(lower, upper)` flow bounds, measured in assignments. We will change selected upper bounds in the two interventions.


In [2]:
# Build the baseline assignment network -
edge_path = joinpath(CHEME5800_L5B_DATA, "Workers-Tasks-Bipartite.edgelist"); # local edge records
source_id = 1; # source node identifier
sink_id = 13; # sink node identifier
baseline_graph = build_sensitivity_graph(edge_path; source = source_id, sink = sink_id);

println("Baseline network: ", length(baseline_graph.nodes), " nodes and ",
    length(baseline_graph.capacity), " directed edges.")


Baseline network: 13 nodes and 23 directed edges.


The supplied network contains 13 nodes and 23 directed edges. Let's compute its maximum flow using Edmonds–Karp, which selects augmenting paths with breadth-first search.

[The `maximumflow(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.maximumflow-Union%7BTuple%7BT%7D%2C%20Tuple%7BT%2C%20MyGraphNodeModel%2C%20MyGraphNodeModel%7D%7D%20where%20T%3C%3AAbstractGraphModel) takes the graph and its source and sink node models. It returns the total flow in `baseline_value` and an edge-flow dictionary in `baseline_flow`. The total counts completed assignments; a dictionary entry gives the flow on one directed edge. Edges absent from the returned dictionary carry zero flow.


In [3]:
# Compute the baseline maximum flow -
baseline_value, baseline_flow = maximumflow(
    baseline_graph, baseline_graph.nodes[source_id], baseline_graph.nodes[sink_id];
    algorithm = EdmondsKarpAlgorithm(),
);
println("Reported baseline maximum flow: ", baseline_value, " assignments")


Reported baseline maximum flow: 3.0 assignments


### Check feasibility and the reported value

The algorithm reports three completed assignments for the supplied network. We now check the returned edge flows independently using [the `validate_sensitivity_flow(...)` function](docs/validate_sensitivity_flow.md). This helper checks three conditions:

* **Edge capacities:** Each flow is nonnegative and does not exceed the edge's upper capacity. The lower bounds in this dataset are all zero.
* **Conservation:** Total inflow equals total outflow at every intermediate node.
* **Source–sink balance:** Net flow leaving the source equals net flow entering the sink.

The report's `valid` field is `true` when all three conditions hold. Its `value` field is the net source outflow calculated from the edge-flow dictionary. We also compare this independently calculated value with `baseline_value`, the total reported by the algorithm. We use an absolute tolerance of $10^{-8}$ assignments for these comparisons to allow for floating-point rounding:


In [4]:
# Independently check the returned flow -
flow_atol = 1e-8; # absolute tolerance for flow comparisons [assignments]
baseline_report = validate_sensitivity_flow(
    baseline_graph, baseline_flow, source_id, sink_id; atol = flow_atol,
);
baseline_value_ok = isapprox(baseline_report.value, baseline_value; atol = flow_atol, rtol = 0);

# Display the feasibility and reported-value checks -
baseline_checks = DataFrame(
    check = ["Edge capacities", "Conservation", "Source–sink balance", "Reported flow value"],
    passed = [baseline_report.capacity_ok, baseline_report.conservation_ok,
        baseline_report.balance_ok, baseline_value_ok],
);
pretty_table(baseline_checks)


┌─────────────────────┬────────┐
│               check │ passed │
│              String │   Bool │
├─────────────────────┼────────┤
│     Edge capacities │   true │
│        Conservation │   true │
│ Source–sink balance │   true │
│ Reported flow value │   true │
└─────────────────────┴────────┘


All four checks pass for the supplied network: the returned flow satisfies the network constraints, and its independently calculated value agrees with the reported total. To establish that three assignments is the maximum, we need an upper bound on the flow that any feasible assignment can achieve.

### Establish optimality with a cut

Consider the source–sink cut with $S=\{1\}$ containing only the source and $T=\mathcal{V}\setminus S$ containing all other nodes, where $\mathcal{V}$ is the network's vertex set. The cut capacity $c(S,T)$ is the sum of the capacities on edges directed from $S$ to $T$. The three source-to-worker edges cross this cut, each with capacity one assignment, so its capacity is given by:

$$
c(S,T)=c(1,2)+c(1,3)+c(1,4)=1+1+1=3.
$$

Every feasible flow satisfies $|f|\leq c(S,T)$, where $|f|$ is its net source outflow. A feasible flow that reaches the cut capacity is therefore maximum. Let's compute this bound from `baseline_graph.capacity` and check that the independently validated flow reaches it:


In [5]:
# Compute the capacity of the cut containing only the source -
baseline_cut_capacity = sum((
    bounds[2] for (edge, bounds) in baseline_graph.capacity
    if edge[1] == source_id && edge[2] != source_id
); init = 0.0); # upper bound on completed assignments

@testset "Baseline feasibility and optimality" begin
    @test baseline_report.valid
    @test baseline_value_ok
    @test isapprox(baseline_report.value, baseline_cut_capacity; atol = flow_atol, rtol = 0)
end;

println("Validated baseline flow: ", baseline_report.value, " assignments")
println("Source-cut capacity: ", baseline_cut_capacity, " assignments")


Test Summary:                       | Pass  Total  Time
Baseline feasibility and optimality |    3      3  0.3s
Validated baseline flow: 3.0 assignments
Source-cut capacity: 3.0 assignments


The validated flow reaches the cut capacity of three assignments, establishing the baseline maximum. Four tasks are available, but the three workers can accept only one assignment each. One task must remain unassigned.

These numerical results apply to the supplied network; changing the edge list or capacities may change the flow value and the cut needed to establish optimality. We will compare each intervention with this same baseline.


### Prediction checkpoint

Before running the next cell: if worker 3 can accept two units from the source, can the downstream network route both units to distinct tasks? Identify the cut that limited the baseline and decide whether that cut has changed.


## Scenario 1: Give worker 3 capacity for a second assignment


In [4]:
expanded_graph = deepcopy(baseline_graph)
expanded_graph.capacity[(1, 3)] = (0.0, 2.0)

expanded_value, expanded_flow = maximumflow(
    expanded_graph, expanded_graph.nodes[1], expanded_graph.nodes[13];
    algorithm = EdmondsKarpAlgorithm(),
)
expanded_report = validate_sensitivity_flow(expanded_graph, expanded_flow, 1, 13)
@test expanded_value == 4.0
@test expanded_report.valid
(value = expanded_value, gain = expanded_value - baseline_value)


(value = 4.0, gain = 1.0)

The source-side cut increases from 3 to 4, and worker 3 has enough distinct outgoing task edges. The fourth task can now be completed.


## Scenario 2: Worker 3 becomes unavailable


In [5]:
outage_graph = deepcopy(baseline_graph)
for edge in keys(outage_graph.capacity)
    if edge[1] == 3
        outage_graph.capacity[edge] = (0.0, 0.0)
    end
end

outage_value, outage_flow = maximumflow(
    outage_graph, outage_graph.nodes[1], outage_graph.nodes[13];
    algorithm = EdmondsKarpAlgorithm(),
)
outage_report = validate_sensitivity_flow(outage_graph, outage_flow, 1, 13)
@test outage_value == 2.0
@test outage_report.valid
(value = outage_value, loss = baseline_value - outage_value)


(value = 2.0, loss = 1.0)

## Compare the interventions


In [6]:
scenario_results = DataFrame(
    scenario = ["baseline", "worker 3 capacity = 2", "worker 3 unavailable"],
    maximum_flow = [baseline_value, expanded_value, outage_value],
    independently_valid = [baseline_report.valid, expanded_report.valid, outage_report.valid],
)
pretty_table(scenario_results)


┌───────────────────────┬──────────────┬─────────────────────┐
│              scenario │ maximum_flow │ independently_valid │
│                String │      Float64 │                Bool │
├───────────────────────┼──────────────┼─────────────────────┤
│              baseline │          3.0 │                true │
│ worker 3 capacity = 2 │          4.0 │                true │
│  worker 3 unavailable │          2.0 │                true │
└───────────────────────┴──────────────┴─────────────────────┘


## Interpretation

- Increasing a capacity helps only if a complete source-to-sink route can use it.
- Removing worker 3 lowers throughput by one because only two source-connected workers remain productive.
- Capacity sensitivity is a network property: inspect cuts and conservation, not just the edited edge.

The next meeting expresses these same conservation and capacity rules as linear constraints.
